# 实验 2：线性回归

<div class="alert alert-warning">
    我们在这里使用基础的 PyTorch 来实现线性回归。然而，在大多数实际应用中，会使用 <code>nn.Module</code> 或 <code>nn.Linear</code> 等抽象概念。
</div>

## 理论概述

$$ H(x) = Wx + b $$

$$ cost(W, b) = \frac{1}{m} \sum^m_{i=1} \left( H(x^{(i)}) - y^{(i)} \right)^2 $$

 - $H(x)$: 对于给定的 $x$ 值，如何进行预测
 - $cost(W, b)$: $H(x)$ 预测 $y$ 的效果如何

## 导入库

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [7]:
# 为了可复现性
torch.manual_seed(1)

## 数据

我们将在此示例中使用伪造数据。

In [8]:
x_train = torch.FloatTensor([[1], [2], [3]])
y_train = torch.FloatTensor([[1], [2], [3]])

In [9]:
print(x_train)
print(x_train.shape)

tensor([[1.],
        [2.],
        [3.]])
torch.Size([3, 1])


In [10]:
print(y_train)
print(y_train.shape)

tensor([[1.],
        [2.],
        [3.]])
torch.Size([3, 1])


默认情况下，PyTorch 使用 NCHW 格式。

## 权重初始化

In [11]:
W = torch.zeros(1, requires_grad=True)
#创建一个长度为 1 的张量，数值为 0
#张量需要参与自动求导（autograd）。
print(W)

tensor([0.], requires_grad=True)


In [12]:
b = torch.zeros(1, requires_grad=True)
print(b)

tensor([0.], requires_grad=True)


## 假设 (Hypothesis)

$$ H(x) = Wx + b $$

In [13]:
hypothesis = x_train * W + b
print(hypothesis)

tensor([[0.],
        [0.],
        [0.]], grad_fn=<AddBackward0>)


## 损失 (Cost)

$$ cost(W, b) = \frac{1}{m} \sum^m_{i=1} \left( H(x^{(i)}) - y^{(i)} \right)^2 $$

In [14]:
print(hypothesis)

tensor([[0.],
        [0.],
        [0.]], grad_fn=<AddBackward0>)


In [15]:
print(y_train)

tensor([[1.],
        [2.],
        [3.]])


In [16]:
print(hypothesis - y_train)

tensor([[-1.],
        [-2.],
        [-3.]], grad_fn=<SubBackward0>)


In [17]:
print((hypothesis - y_train) ** 2)

tensor([[1.],
        [4.],
        [9.]], grad_fn=<PowBackward0>)


In [18]:
cost = torch.mean((hypothesis - y_train) ** 2)
print(cost)

tensor(4.6667, grad_fn=<MeanBackward0>)


## 梯度下降

In [19]:
optimizer = optim.SGD([W, b], lr=0.01)

In [20]:
optimizer.zero_grad()
cost.backward()
optimizer.step()

In [21]:
print(W)
print(b)

tensor([0.0933], requires_grad=True)
tensor([0.0400], requires_grad=True)


让我们检查一下假设现在是否更好。

In [22]:
hypothesis = x_train * W + b
print(hypothesis)

tensor([[0.1333],
        [0.2267],
        [0.3200]], grad_fn=<AddBackward0>)


In [23]:
cost = torch.mean((hypothesis - y_train) ** 2)
print(cost)

tensor(3.6927, grad_fn=<MeanBackward0>)


## 使用完整代码训练

实际上，我们将在数据集上进行多次 epoch 的训练。这可以通过简单的循环来完成。

In [24]:
# 数据
x_train = torch.FloatTensor([[1], [2], [3]])
y_train = torch.FloatTensor([[1], [2], [3]])
# 模型初始化
W = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
# 优化器设置
optimizer = optim.SGD([W, b], lr=0.01)

nb_epochs = 1000
for epoch in range(nb_epochs + 1):
    
    # 计算 H(x)
    hypothesis = x_train * W + b
    
    # 计算 cost
    cost = torch.mean((hypothesis - y_train) ** 2)

    # 通过 cost 改善 H(x)
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()

    # 每 100 次打印日志
    if epoch % 100 == 0:
        print('Epoch {:4d}/{} W: {:.3f}, b: {:.3f} Cost: {:.6f}'.format(
            epoch, nb_epochs, W.item(), b.item(), cost.item()
        ))

Epoch    0/1000 W: 0.093, b: 0.040 Cost: 4.666667
Epoch  100/1000 W: 0.873, b: 0.289 Cost: 0.012043
Epoch  200/1000 W: 0.900, b: 0.227 Cost: 0.007442
Epoch  300/1000 W: 0.921, b: 0.179 Cost: 0.004598
Epoch  400/1000 W: 0.938, b: 0.140 Cost: 0.002842
Epoch  500/1000 W: 0.951, b: 0.110 Cost: 0.001756
Epoch  600/1000 W: 0.962, b: 0.087 Cost: 0.001085
Epoch  700/1000 W: 0.970, b: 0.068 Cost: 0.000670
Epoch  800/1000 W: 0.976, b: 0.054 Cost: 0.000414
Epoch  900/1000 W: 0.981, b: 0.042 Cost: 0.000256
Epoch 1000/1000 W: 0.985, b: 0.033 Cost: 0.000158


## 使用 `nn.Module` 的高级实现

记得我们有这个伪造数据。

In [25]:
x_train = torch.FloatTensor([[1], [2], [3]])
y_train = torch.FloatTensor([[1], [2], [3]])

现在我们需要创建一个线性回归模型，默认情况下，PyTorch 中的所有模型都是通过继承提供的 `nn.Module` 来创建的。

In [26]:
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)
    
    def forward(self, x):
        return self.linear(x)

在模型的 `__init__` 中，我们需要定义要使用的层。因为我们这里要创建一个线性回归模型，所以我们将使用 `nn.Linear`。然后在 `forward` 中，我们定义模型如何从输入值计算输出值。

In [27]:
model = LinearRegressionModel()

## 假设 (Hypothesis)

现在让我们创建模型并计算预测值 $H(x)$

In [28]:
hypothesis = model(x_train)

In [29]:
print(hypothesis)

tensor([[0.0739],
        [0.5891],
        [1.1044]], grad_fn=<AddmmBackward0>)


## 损失 (Cost)

现在让我们使用均方误差 (MSE) 来计算成本。PyTorch 也默认提供了 MSE。

In [30]:
print(hypothesis)
print(y_train)

tensor([[0.0739],
        [0.5891],
        [1.1044]], grad_fn=<AddmmBackward0>)
tensor([[1.],
        [2.],
        [3.]])


In [31]:
cost = F.mse_loss(hypothesis, y_train)

In [32]:
print(cost)

tensor(2.1471, grad_fn=<MseLossBackward0>)


## 梯度下降

最后，我们利用给定的 cost 来调整 $H(x)$ 的 $W, b$ 以降低 cost。这时可以使用 PyTorch 的 `torch.optim` 中提供的 `optimizer` 之一。

In [33]:
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [34]:
optimizer.zero_grad()
cost.backward()
optimizer.step()

## 使用完整代码训练

既然我们已经理解了线性回归的代码，现在让我们实际运行代码来进行拟合。

In [35]:
# 数据
x_train = torch.FloatTensor([[1], [2], [3]])
y_train = torch.FloatTensor([[1], [2], [3]])
# 模型初始化
model = LinearRegressionModel()
# 优化器设置
optimizer = optim.SGD(model.parameters(), lr=0.01)

nb_epochs = 1000
for epoch in range(nb_epochs + 1):
    
    # 计算 H(x)
    prediction = model(x_train)
    
    # 计算 cost
    cost = F.mse_loss(prediction, y_train)
    
    # 通过 cost 改善 H(x)
    optimizer.zero_grad()
    cost.backward()
    optimizer.step()
    
    # 每 100 次打印日志
    if epoch % 100 == 0:
        params = list(model.parameters())
        W = params[0].item()
        b = params[1].item()
        print('Epoch {:4d}/{} W: {:.3f}, b: {:.3f} Cost: {:.6f}'.format(
            epoch, nb_epochs, W, b, cost.item()
        ))

Epoch    0/1000 W: -0.101, b: 0.508 Cost: 4.630286
Epoch  100/1000 W: 0.713, b: 0.653 Cost: 0.061555
Epoch  200/1000 W: 0.774, b: 0.514 Cost: 0.038037
Epoch  300/1000 W: 0.822, b: 0.404 Cost: 0.023505
Epoch  400/1000 W: 0.860, b: 0.317 Cost: 0.014525
Epoch  500/1000 W: 0.890, b: 0.250 Cost: 0.008975
Epoch  600/1000 W: 0.914, b: 0.196 Cost: 0.005546
Epoch  700/1000 W: 0.932, b: 0.154 Cost: 0.003427
Epoch  800/1000 W: 0.947, b: 0.121 Cost: 0.002118
Epoch  900/1000 W: 0.958, b: 0.095 Cost: 0.001309
Epoch 1000/1000 W: 0.967, b: 0.075 Cost: 0.000809


可以看到，通过逐渐调整 $H(x)$ 的 $W$ 和 $b$，cost 正在减小。